In [1]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path
from tabulate import tabulate

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class

In [2]:

# define dataset path
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'

project_name = 'HCFC1' #change here for different task name
task_name = 'Dataset002_' + project_name 

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

In [3]:
def read_nifti(path):
    img = nib.load(path)

    return img.get_fdata(),img.shape

In [4]:
def count_pixels_in_regions(nifti_file):
    region_values = list(range(25))
    # Load the NIfTI file
    img = nib.load(nifti_file)
    data = img.get_fdata()
    
    # Initialize a list to store the counts
    pixel_counts = [0] * len(region_values)
    
    # Count the pixels in each region
    for idx, value in enumerate(region_values):
        pixel_counts[idx] = np.sum(data == value)
    
    return pixel_counts


In [12]:
region_pixels = []

# Specify the path to your JSON file
json_file_path = TASK_PATH / 'dataset.json'

# Read the JSON file
with open(json_file_path, 'r') as file:
    data = json.load(file)

labels = data['labels']
head = ['Fold'] + ['Volume ID'] + list(labels.keys())

region_pixels.append(head)

dataset_config_path = f"{task_name}/nnUNetTrainer__nnUNetPlans__3d_fullres"
gtPath = GT_TRAINING_DATASET_PATH
for i in range(5):
    
    imagePath = f"/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/{dataset_config_path}/fold_{i}/validation"

    all_files = os.listdir(imagePath)
    for file in all_files:
        if file.endswith(".nii.gz"):
            imgP = os.path.join(imagePath, file)
            gtP = os.path.join(gtPath, file)
            # print(imgP)
            # print(gtP)
            # imgData, i_ = read_nifti(imgP)
            # gtData, g_ = read_nifti(gtP)
            newname = file.split("_")[0]
            print(newname)
            # print(i_)
            # print(g_)

            concatenated_array = np.concatenate(([i],[newname], count_pixels_in_regions(gtP)))
   
            concatenated_array = np.transpose(concatenated_array)

            region_pixels.append(concatenated_array)


NG4108
NG4111
NG4116
NG4119
NG4115
NG4120
NG4114
NG4117
NG4109
NG4112
NG4110
NG4113


In [13]:
print(tabulate(region_pixels, tablefmt="grid"))

+------+-----------+------------+----------+---------+---------+--------+---------+---------+--------+-------+--------+--------+--------+--------+--------+--------+-------+-------+---------+---------+---------+---------+---------+---------+---------+---------+
| Fold | Volume ID | background | CTX+     | cc+     | CPu     | DG     | HP      | RHP     | A      | ig    | fi     | ac     | ic     | st     | f      | och    | fr    | Hb    | TH      | HY      | MB      | P       | MY      | TCB     | V       | OB      |
+------+-----------+------------+----------+---------+---------+--------+---------+---------+--------+-------+--------+--------+--------+--------+--------+--------+-------+-------+---------+---------+---------+---------+---------+---------+---------+---------+
| 0    | NG4108    | 100013582  | 13123790 | 908045  | 5677092 | 686849 | 1506707 | 1297884 | 250754 | 10214 | 272023 | 109093 | 365518 | 77500  | 66535  | 80672  | 19389 | 86286 | 2087008 | 1445082 | 3766019 | 242723

In [14]:
def transpose_table(table):
    transposed_table = []

    # Get the header row and remove it from the table
    header = table.pop(0)
    # Initialize transposed table with the Volume ID column
    transposed_table.append(["Fold"] + [row[0] for row in table])

    # Transpose the table
    for i in range(1, len(header)):
        transposed_row = [header[i]]
        for j in range(len(table)):
            transposed_row.append(table[j][i])
        transposed_table.append(transposed_row)

    return transposed_table


# # Transpose the table
transposed_table = transpose_table(region_pixels)

print(tabulate(transposed_table, tablefmt="grid"))


+------------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| Fold       | 0         | 0        | 0        | 1        | 1        | 1        | 2        | 2        | 3        | 3        | 4        | 4        |
+------------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| Volume ID  | NG4108    | NG4111   | NG4116   | NG4119   | NG4115   | NG4120   | NG4114   | NG4117   | NG4109   | NG4112   | NG4110   | NG4113   |
+------------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| background | 100013582 | 52176648 | 94258667 | 59210298 | 70221045 | 77420685 | 52349698 | 55401809 | 61268942 | 47907029 | 98251432 | 82188426 |
+------------+-----------+----------+----------+----------+----------+----------+----------+----------+---------

In [24]:
import pandas as pd

# Define your array
data = transpose_table(region_pixels) 

# Define column names
columns = ["Volume ID"] + [f"Value_{i}" for i in range(1, len(transposed_table[0]))]

# Create DataFrame
df = pd.DataFrame(transposed_table)

# Save DataFrame to Excel
df.to_excel(BASE_PATH/"data/region_pixels_6.xlsx", index=True)

print("DataFrame saved to region_pixels.xlsx")


DataFrame saved to region_pixels.xlsx


In [25]:
df

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Fold,0,0,0,1,1,1,2,2,3,3,4,4
1,Volume ID,NG4108,NG4111,NG4116,NG4119,NG4115,NG4120,NG4114,NG4117,NG4109,NG4112,NG4110,NG4113
2,background,100013582,52176648,94258667,59210298,70221045,77420685,52349698,55401809,61268942,47907029,98251432,82188426
3,CTX+,13123790,12031753,13780858,11017591,15015588,10144349,9309837,11340841,15066101,13047163,14989756,13846545
4,cc+,908045,988205,999139,728859,1061451,809458,737164,901475,1030712,1103185,1130853,1071331
5,CPu,5677092,5888111,5242927,4939545,5739518,5731060,5049717,5393050,5701843,5799002,5882564,5793639
6,DG,686849,721753,597888,536471,671236,795151,597663,590658,671340,700306,652725,713284
7,HP,1506707,1580086,1460669,1310951,1706985,1362181,1423286,1425295,1611106,1734336,1810564,1818336
8,RHP,1297884,1208729,1594038,1240692,1722468,1143885,1148011,1382542,1603376,1648532,1654333,1435451
9,A,250754,182951,284201,189176,300290,172005,664,224049,238222,274419,274820,282921
